In [6]:
from __future__ import annotations

import argparse
import html
import json
import re
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from urllib.error import HTTPError, URLError
from urllib.parse import urljoin
from urllib.request import Request, urlopen


BASE_URL = "https://ev-database.org/"
CATALOG_URL = BASE_URL
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) EVDatabaseScraper/1.0"
REQUEST_TIMEOUT = 30
REQUEST_RETRIES = 5
REQUEST_BACKOFF_SECONDS = 2.0
CAR_CARD_TOKEN = '<div class="list-item" data-jplist-item>'


def fetch_html(url: str) -> str:
    request = Request(
        url,
        headers={
            "User-Agent": USER_AGENT,
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9",
        },
    )

    last_error: Exception | None = None
    for attempt in range(REQUEST_RETRIES):
        try:
            with urlopen(request, timeout=REQUEST_TIMEOUT) as response:
                return response.read().decode("utf-8", "replace")
        except HTTPError as error:
            last_error = error
            retryable = error.code in {429, 500, 502, 503, 504}
            if not retryable or attempt == REQUEST_RETRIES - 1:
                raise
        except URLError as error:
            last_error = error
            if attempt == REQUEST_RETRIES - 1:
                raise

        time.sleep(REQUEST_BACKOFF_SECONDS * (attempt + 1))

    if last_error is not None:
        raise last_error

    raise RuntimeError(f"Unable to fetch {url}")


def strip_tags(value: str | None) -> str:
    if value is None:
        return ""
    value = html.unescape(value)
    value = re.sub(r"<[^>]+>", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def extract_first(pattern: str, text: str, flags: int = re.S) -> str:
    match = re.search(pattern, text, flags)
    return match.group(1).strip() if match else ""


def parse_number(text: str | None) -> float | None:
    if not text:
        return None
    match = re.search(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return float(match.group(0)) if match else None


def parse_int(text: str | None) -> int | None:
    number = parse_number(text)
    return int(number) if number is not None else None


def normalize_key(value: str) -> str:
    value = strip_tags(value)
    value = value.lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


def parse_title(title_html: str) -> dict[str, str]:
    brand = extract_first(r'<span class="[^"]+">([^<]+)</span>', title_html)
    model = extract_first(r'<span class="model">(.*?)</span>', title_html)
    canonical = extract_first(r'<span class="hidden">([^<]+)</span>', title_html)
    display_name = strip_tags(title_html)

    if not canonical:
        canonical = display_name

    return {
        "brand": brand,
        "model": model,
        "canonical_name": canonical,
        "display_name": display_name,
    }


def parse_card(card_html: str) -> dict[str, Any]:
    href = extract_first(r'<a href="([^"]+)"\s+class="title">', card_html)
    title_html = extract_first(r'<a href="[^"]+"\s+class="title">(.*?)</a>', card_html)
    title = parse_title(title_html)

    car_id_match = re.search(r"/car/(\d+)/", href)
    car_id = int(car_id_match.group(1)) if car_id_match else None

    summary = {
        "range_km": parse_number(extract_first(r'<span class="erange_real">([^<]+)</span>', card_html)),
        "efficiency_wh_per_km": parse_number(extract_first(r'<span class="efficiency">([^<]+)</span>', card_html)),
        "weight_kg": parse_int(extract_first(r'<span class="weight_p">([^<]+)</span>', card_html)),
        "acceleration_sec": parse_number(extract_first(r'<span class="acceleration_p">([^<]+)</span>', card_html)),
        "one_stop_range_km": parse_number(extract_first(r'<span class="long_distance_total">([^<]+)</span>', card_html)),
        "battery_kwh": parse_number(extract_first(r'<span class="battery_p">([^<]+)</span>', card_html)),
        "fastcharge_kw": parse_number(extract_first(r'<span class="fastcharge_speed_print">([^<]+)</span>', card_html)),
        "towing_kg": parse_int(extract_first(r'<span class="towweight_p">([^<]+)</span>', card_html)),
        "cargo_volume_l": parse_int(extract_first(r'<span class="cargo">([^<]+)</span>', card_html)),
        "price_per_range_eur_per_km": parse_number(extract_first(r'<span class="priceperrange_p">([^<]+)</span>', card_html)),
    }

    availability = extract_first(r'<div class="availability[^>]*">([^<]+)</div>', card_html)
    drive_type = extract_first(r'data-tooltip="([^"]+Wheel Drive)"', card_html)
    segment_letter = extract_first(r'data-tooltip="Market Segment"\s+class="size-[^"]+">([A-Z])</span>', card_html)
    seats = parse_int(extract_first(r'data-tooltip="Number of seats"[^>]*>\s*(?:<i[^>]*></i>\s*)?<span>(\d+)</span>', card_html))

    prices = {
        "de": extract_first(r'<span class="country_de"[^>]*>([^<]+)</span>', card_html),
        "nl": extract_first(r'<span class="country_nl"[^>]*>([^<]+)</span>', card_html),
        "uk": extract_first(r'<span class="country_uk"[^>]*>([^<]+)</span>', card_html),
    }
    prices = {code: value for code, value in prices.items() if value}

    return {
        "car_id": car_id,
        "url": urljoin(BASE_URL, href),
        **title,
        "availability": availability,
        "drive_type": drive_type,
        "segment_letter": segment_letter,
        "seats": seats,
        "summary": summary,
        "prices": prices,
    }


def parse_section_tables(html_text: str) -> dict[str, dict[str, str]]:
    sections: dict[str, dict[str, str]] = {}
    heading_matches = list(re.finditer(r"<h2>(.*?)</h2>", html_text, re.S))

    for index, heading_match in enumerate(heading_matches):
        section_name = normalize_key(heading_match.group(1))
        section_start = heading_match.end()
        section_end = heading_matches[index + 1].start() if index + 1 < len(heading_matches) else len(html_text)
        section_html = html_text[section_start:section_end]

        rows: dict[str, str] = {}
        for row_html in re.findall(r"<tr>(.*?)</tr>", section_html, re.S):
            cells = [strip_tags(cell) for cell in re.findall(r"<t[dh]>(.*?)</t[dh]>", row_html, re.S)]
            if len(cells) == 2:
                key, value = cells
                rows[key] = value

        if rows:
            sections[section_name] = rows

    return sections


def parse_detail_page(detail_html: str) -> dict[str, Any]:
    title = strip_tags(extract_first(r"<h1[^>]*>(.*?)</h1>", detail_html))
    sections = parse_section_tables(detail_html)

    misc = sections.get("miscellaneous", {})
    battery = sections.get("battery", {})
    performance = sections.get("performance", {})
    dimensions = sections.get("dimensions_and_weight", {})
    energy = sections.get("energy_consumption", {})

    def section_value(section: dict[str, str], *labels: str) -> str:
        for label in labels:
            if label in section:
                return section[label]
        return ""

    return {
        "page_title": title,
        "body_style": section_value(misc, "Car Body"),
        "segment": section_value(misc, "Segment"),
        "seat_count": parse_int(section_value(misc, "Seats")),
        "platform": section_value(misc, "Platform"),
        "ev_dedicated_platform": section_value(misc, "EV Dedicated Platform"),
        "roof_rails": section_value(misc, "Roof Rails"),
        "heat_pump": section_value(misc, "Heat pump (HP)"),
        "hp_standard_equipment": section_value(misc, "HP Standard Equipment"),
        "battery": {
            "nominal_capacity_kwh": parse_number(section_value(battery, "Nominal Capacity *", "Nominal Capacity*", "Nominal Capacity")),
            "usable_capacity_kwh": parse_number(section_value(battery, "Useable Capacity*", "Useable Capacity *", "Useable Capacity")),
            "battery_type": section_value(battery, "Battery Type"),
            "architecture_v": parse_number(section_value(battery, "Architecture")),
            "warranty_period": section_value(battery, "Warranty Period"),
            "warranty_mileage_km": parse_number(section_value(battery, "Warranty Mileage")),
            "cathode_material": section_value(battery, "Cathode Material"),
            "pack_configuration": section_value(battery, "Pack Configuration"),
            "nominal_voltage": section_value(battery, "Nominal Voltage"),
            "form_factor": section_value(battery, "Form Factor"),
        },
        "performance": {
            "acceleration_0_100_sec": parse_number(section_value(performance, "Acceleration 0 - 100 km/h")),
            "top_speed_kmh": parse_number(section_value(performance, "Top Speed")),
            "electric_range_km": parse_number(section_value(performance, "Electric Range *", "Electric Range")),
            "total_power_kw": parse_number(section_value(performance, "Total Power")),
            "total_torque_nm": parse_number(section_value(performance, "Total Torque")),
            "drive": section_value(performance, "Drive"),
        },
        "energy_consumption": {
            "evdb_real_range_km": parse_number(section_value(energy, "Range *", "Range")),
            "vehicle_consumption_wh_per_km": parse_number(section_value(energy, "Vehicle Consumption *", "Vehicle Consumption")),
            "co2_emissions_g_per_km": parse_number(section_value(energy, "CO2 Emissions")),
            "vehicle_fuel_equivalent_l_per_100km": section_value(energy, "Vehicle Fuel Equivalent *", "Vehicle Fuel Equivalent"),
        },
        "dimensions_and_weight": {
            "length_mm": parse_number(section_value(dimensions, "Length")),
            "width_mm": parse_number(section_value(dimensions, "Width")),
            "width_with_mirrors_mm": parse_number(section_value(dimensions, "Width with mirrors")),
            "height_mm": parse_number(section_value(dimensions, "Height")),
            "wheelbase_mm": parse_number(section_value(dimensions, "Wheelbase")),
            "weight_unladen_eu_kg": parse_number(section_value(dimensions, "Weight Unladen (EU)")),
            "gross_vehicle_weight_kg": parse_number(section_value(dimensions, "Gross Vehicle Weight (GVWR)")),
            "max_payload_kg": parse_number(section_value(dimensions, "Max. Payload")),
            "cargo_volume_l": parse_number(section_value(dimensions, "Cargo Volume")),
            "cargo_volume_max_l": parse_number(section_value(dimensions, "Cargo Volume Max")),
            "cargo_volume_frunk_l": parse_number(section_value(dimensions, "Cargo Volume Frunk")),
            "roof_load_kg": parse_number(section_value(dimensions, "Roof Load")),
            "tow_hitch_possible": section_value(dimensions, "Tow Hitch Possible"),
            "towing_weight_unbraked_kg": parse_number(section_value(dimensions, "Towing Weight Unbraked")),
            "towing_weight_braked_kg": parse_number(section_value(dimensions, "Towing Weight Braked")),
            "vertical_load_max_kg": parse_number(section_value(dimensions, "Vertical Load Max")),
        },
        "raw_sections": sections,
    }


def build_catalog_dataset(limit: int | None = None) -> list[dict[str, Any]]:
    catalog_html = fetch_html(CATALOG_URL)
    cards = []

    for card_html in catalog_html.split(CAR_CARD_TOKEN)[1:]:
        card = parse_card(CAR_CARD_TOKEN + card_html)
        if card.get("car_id") is not None:
            cards.append(card)

    if limit is not None:
        cards = cards[:limit]

    vehicles: list[dict[str, Any]] = []
    for card in cards:
        vehicles.append(
            {
                "car_id": card["car_id"],
                "source_url": card["url"],
                "brand": card["brand"],
                "model": card["model"],
                "canonical_name": card["canonical_name"],
                "display_name": card["display_name"],
                "availability": card["availability"],
                "drive_type": card["drive_type"],
                "segment_letter": card["segment_letter"],
                "seats": card["seats"],
                "summary": card["summary"],
                "prices": card["prices"],
            }
        )

    return vehicles


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


output_path = Path.cwd() / "project_ev" / "ev_database_vehicles.json"
vehicles = build_catalog_dataset()
payload = {
    "source": BASE_URL,
    "scraped_at": datetime.now(timezone.utc).isoformat(),
    "count": len(vehicles),
    "vehicles": vehicles,
}
write_json(output_path, payload)

print(f"Wrote {len(vehicles)} vehicles to {output_path}")

Wrote 1301 vehicles to c:\Users\tomde\OneDrive\Documentatie - professioneel - opleiding\AI pro 2025-26\Project - Gen AI\project_ev\ev_database_vehicles.json


In [6]:
# Data flattening, chunking, embedding and storage example
# - Flattens vehicle JSON entries
# - Chunks text per vehicle into ~800-char pieces
# - Shows one example chunk
# - Embeds with local Ollama model nomic-embed-text and stores in Chroma

import os
import json
import time
import textwrap
from typing import Any, Dict, List

try:
    import requests
except Exception:
    print("Missing dependency: requests. Install with: pip install requests")
    raise

try:
    import chromadb
except Exception:
    print("Missing dependency: chromadb. Install with: pip install chromadb")
    raise

def flatten_dict(d: Dict[str, Any], parent_key: str = "") -> Dict[str, Any]:
    items: Dict[str, Any] = {}
    for k, v in d.items():
        new_key = f"{parent_key}.{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten_dict(v, new_key))
        else:
            items[new_key] = v
    return items

def vehicle_to_text(vehicle: Dict[str, Any]) -> str:
    flat = flatten_dict(vehicle)
    lines = [f"{k}: {flat[k]}" for k in sorted(flat.keys())]
    return "\n".join(lines)

def chunk_text(text: str, max_chars: int = 800) -> List[str]:
    if len(text) <= max_chars:
        return [text]
    paragraphs = text.split("\n\n")
    chunks: List[str] = []
    current: List[str] = []
    current_len = 0
    for p in paragraphs:
        p = p.strip()
        if not p:
            continue
        if current_len + len(p) + 2 <= max_chars:
            current.append(p)
            current_len += len(p) + 2
        else:
            if current:
                chunks.append("\n\n".join(current))
            if len(p) > max_chars:
                for sub in textwrap.wrap(p, width=max_chars, break_long_words=False):
                    chunks.append(sub)
                current = []
                current_len = 0
            else:
                current = [p]
                current_len = len(p) + 2
    if current:
        chunks.append("\n\n".join(current))
    # Safety split any remaining oversized chunks.
    final: List[str] = []
    for c in chunks:
        if len(c) <= max_chars:
            final.append(c)
        else:
            final.extend(textwrap.wrap(c, width=max_chars, break_long_words=False))
    return final

def get_embedding_ollama(
    text: str,
    model: str = "nomic-embed-text",
    base_url: str = "http://localhost:11434",
) -> List[float]:
    """Get an embedding from local Ollama.

    Supports both endpoints:
    - /api/embeddings (older/common)
    - /api/embed (newer)
    """
    # Try /api/embeddings first.
    url_embeddings = f"{base_url.rstrip('/')}/api/embeddings"
    payload = {"model": model, "prompt": text}
    resp = requests.post(url_embeddings, json=payload, timeout=60)
    if resp.ok:
        data = resp.json()
        if "embedding" in data and isinstance(data["embedding"], list):
            return data["embedding"]

    # Fallback to /api/embed.
    url_embed = f"{base_url.rstrip('/')}/api/embed"
    payload2 = {"model": model, "input": text}
    resp2 = requests.post(url_embed, json=payload2, timeout=60)
    resp2.raise_for_status()
    data2 = resp2.json()
    if "embeddings" in data2 and isinstance(data2["embeddings"], list) and data2["embeddings"]:
        return data2["embeddings"][0]
    raise RuntimeError(f"Unexpected Ollama embedding response: {data2}")

# Load JSON file from repo root
json_path = "ev_database_vehicles.json"
with open(json_path, "r", encoding="utf-8") as f:
    payload = json.load(f)
vehicles = payload.get("vehicles", payload)
print(f"Loaded {len(vehicles)} vehicles from {json_path}")


# Build chunks for the first vehicle and show an example
first = vehicles[0] if vehicles else {}
first_id = str(first.get("car_id") or first.get("id") or "vehicle_0")
text = vehicle_to_text(first)
chunks = chunk_text(text, max_chars=800)
print("Example vehicle id:", first_id)
print(f"Number of chunks for first vehicle: {len(chunks)}")
print("--- Example chunk (truncated to 1000 chars) ---")
print(chunks[0][:1000] if chunks else "No chunks generated")

# Embed with local Ollama and store in Chroma.
# Ensure the model exists locally: ollama pull nomic-embed-text
ollama_base_url = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
ollama_model = os.environ.get("OLLAMA_EMBED_MODEL", "nomic-embed-text")
max_vehicles = len(vehicles)  # Set to a smaller number for testing, e.g. 10

ids, docs, metadatas, embeddings = [], [], [], []
for idx, vehicle in enumerate(vehicles[:max_vehicles]):
    vid = str(vehicle.get("car_id") or vehicle.get("id") or f"vehicle_{idx}")
    text = vehicle_to_text(vehicle)
    vchunks = chunk_text(text, max_chars=800)
    for i, c in enumerate(vchunks):
        cid = f"{vid}::chunk::{i}"
        emb = get_embedding_ollama(c, model=ollama_model, base_url=ollama_base_url)
        ids.append(cid)
        docs.append(c)
        metadatas.append({"vehicle_id": vid, "chunk_index": i})
        embeddings.append(emb)
        time.sleep(0.01)

client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="ev_vehicles", metadata={"source": json_path, "embed_model": ollama_model})
collection.add(ids=ids, documents=docs, metadatas=metadatas, embeddings=embeddings)
try:
    client.persist()
except Exception:
    pass

print(
    f"Embedded and stored {len(ids)} chunks for {min(len(vehicles), max_vehicles)} vehicles "
    f"in Chroma collection 'ev_vehicles' using Ollama model '{ollama_model}'."
)

Loaded 1301 vehicles from ev_database_vehicles.json
--- First 20 flattened records ---

Record 1:
{
  "car_id": 3403,
  "source_url": "https://ev-database.org/car/3403/Tesla-Model-3-RWD",
  "brand": "Tesla",
  "model": "Model 3 RWD <span style='font-size: 13px;'>(Highland)",
  "canonical_name": "Tesla Model 3 RWD",
  "display_name": "Tesla Model 3 RWD (Highland) Tesla Model 3 RWD",
  "availability": "Available to order since December 2025",
  "drive_type": "Rear Wheel Drive",
  "segment_letter": "",
  "seats": 5,
  "summary.range_km": 445.0,
  "summary.efficiency_wh_per_km": 135.0,
  "summary.weight_kg": 1847,
  "summary.acceleration_sec": 6.2,
  "summary.one_stop_range_km": 523.0,
  "summary.battery_kwh": 60.0,
  "summary.fastcharge_kw": 110.0,
  "summary.towing_kg": 1000,
  "summary.cargo_volume_l": 682,
  "summary.price_per_range_eur_per_km": 89.0,
  "prices.de": "€37,970",
  "prices.nl": "€36,990",
  "prices.uk": "£37,990"
}

Record 2:
{
  "car_id": 3290,
  "source_url": "https://e

In [7]:
# Display the first 20 flattened JSON records.
first_20_flattened = [flatten_dict(v) for v in vehicles[:20]]
print("--- First 20 flattened records ---")
for rec_idx, rec in enumerate(first_20_flattened, start=1):
    print(f"\nRecord {rec_idx}:")
    print(json.dumps(rec, ensure_ascii=False, indent=2))


--- First 20 flattened records ---

Record 1:
{
  "car_id": 3403,
  "source_url": "https://ev-database.org/car/3403/Tesla-Model-3-RWD",
  "brand": "Tesla",
  "model": "Model 3 RWD <span style='font-size: 13px;'>(Highland)",
  "canonical_name": "Tesla Model 3 RWD",
  "display_name": "Tesla Model 3 RWD (Highland) Tesla Model 3 RWD",
  "availability": "Available to order since December 2025",
  "drive_type": "Rear Wheel Drive",
  "segment_letter": "",
  "seats": 5,
  "summary.range_km": 445.0,
  "summary.efficiency_wh_per_km": 135.0,
  "summary.weight_kg": 1847,
  "summary.acceleration_sec": 6.2,
  "summary.one_stop_range_km": 523.0,
  "summary.battery_kwh": 60.0,
  "summary.fastcharge_kw": 110.0,
  "summary.towing_kg": 1000,
  "summary.cargo_volume_l": 682,
  "summary.price_per_range_eur_per_km": 89.0,
  "prices.de": "€37,970",
  "prices.nl": "€36,990",
  "prices.uk": "£37,990"
}

Record 2:
{
  "car_id": 3290,
  "source_url": "https://ev-database.org/car/3290/BMW-iX3-50-xDrive",
  "brand

In [15]:
# Retrieval assistant: ask user, query top-10, show similarity, confirm brand/model, and fetch complete chunks
import json
import re
from collections import defaultdict
from typing import Any, Dict, List
import requests
import chromadb

OLLAMA_BASE_URL = "http://localhost:11434"
EMBED_MODEL = "nomic-embed-text"
CHAT_MODEL = "llama3"
COLLECTION_NAME = "ev_vehicles"

def safe_input(prompt: str, default: str) -> str:
    """Use interactive input when available, otherwise fallback to default."""
    try:
        value = input(prompt).strip()
        return value if value else default
    except EOFError:
        print(f"[No interactive input detected] Using default: {default}")
        return default

def ollama_embed(text: str, model: str = EMBED_MODEL, base_url: str = OLLAMA_BASE_URL) -> List[float]:
    # First try the common endpoint.
    url1 = f"{base_url.rstrip('/')}/api/embeddings"
    r1 = requests.post(url1, json={"model": model, "prompt": text}, timeout=60)
    if r1.ok:
        d1 = r1.json()
        if "embedding" in d1 and isinstance(d1["embedding"], list):
            return d1["embedding"]

    # Fallback for newer endpoint shape.
    url2 = f"{base_url.rstrip('/')}/api/embed"
    r2 = requests.post(url2, json={"model": model, "input": text}, timeout=60)
    r2.raise_for_status()
    d2 = r2.json()
    if "embeddings" in d2 and isinstance(d2["embeddings"], list) and d2["embeddings"]:
        return d2["embeddings"][0]
    raise RuntimeError(f"Unexpected Ollama embedding response: {d2}")

def ollama_chat(prompt: str, model: str = CHAT_MODEL, base_url: str = OLLAMA_BASE_URL) -> str:
    # Try /api/chat first.
    try:
        url_chat = f"{base_url.rstrip('/')}/api/chat"
        payload_chat = {
            "model": model,
            "stream": False,
            "messages": [
                {"role": "system", "content": "You are a concise EV search assistant."},
                {"role": "user", "content": prompt},
            ],
        }
        resp_chat = requests.post(url_chat, json=payload_chat, timeout=120)
        if resp_chat.ok:
            data_chat = resp_chat.json()
            return data_chat.get("message", {}).get("content", "")
    except Exception:
        pass

    # Fallback for older Ollama versions: /api/generate.
    try:
        url_gen = f"{base_url.rstrip('/')}/api/generate"
        payload_gen = {"model": model, "prompt": prompt, "stream": False}
        resp_gen = requests.post(url_gen, json=payload_gen, timeout=120)
        if resp_gen.ok:
            data_gen = resp_gen.json()
            return data_gen.get("response", "")
    except Exception:
        pass

    # Fallback for OpenAI-compatible local servers.
    try:
        url_openai = f"{base_url.rstrip('/')}/v1/chat/completions"
        payload_openai = {
            "model": model,
            "messages": [
                {"role": "system", "content": "You are a concise EV search assistant."},
                {"role": "user", "content": prompt},
            ],
            "temperature": 0.2,
        }
        resp_openai = requests.post(url_openai, json=payload_openai, timeout=120)
        if resp_openai.ok:
            data_openai = resp_openai.json()
            choices = data_openai.get("choices", [])
            if choices:
                return choices[0].get("message", {}).get("content", "")
    except Exception:
        pass

    print("Warning: llama3 endpoint not reachable. Falling back to direct keyword query.")
    return ""

def extract_brand_model(text: str) -> tuple[str, str]:
    brand_match = re.search(r"^brand:\s*(.+)$", text, flags=re.M)
    model_match = re.search(r"^model:\s*(.+)$", text, flags=re.M)
    brand = brand_match.group(1).strip() if brand_match else "unknown"
    model = model_match.group(1).strip() if model_match else "unknown"
    return brand, model

def distance_to_similarity(distance: float) -> float:
    # Works across distance types; higher is better and within (0,1].
    return 1.0 / (1.0 + max(distance, 0.0))

# 1) Ask user for input
user_car_description = safe_input(
    "Describe your car (model, range, battery, drive type, etc.): ",
    "I have an electric sedan with about 445 km range and rear wheel drive.",
)
user_purchase_time = safe_input(
    "When did you buy it? (month/year): ",
    "December 2025",
)
user_brand_guess = safe_input(
    "Which brand do you think it is?: ",
    "Tesla",
)

# 2) Use llama3 to build a retrieval-focused search query
query_prompt = f"""
Create one concise semantic search query to find a matching EV in a vector database.
Include key facts from the user.

User description: {user_car_description}
Purchase time: {user_purchase_time}
Brand guess: {user_brand_guess}
""".strip()

search_query = ollama_chat(query_prompt, model=CHAT_MODEL).strip()
if not search_query:
    search_query = f"{user_car_description} bought {user_purchase_time} brand {user_brand_guess}"

print("\nSearch query used:")
print(search_query)

# 3) Retrieve top-10 from vector DB
query_embedding = ollama_embed(search_query, model=EMBED_MODEL)
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_collection(COLLECTION_NAME)

res = collection.query(
    query_embeddings=[query_embedding],
    n_results=10,
    include=["documents", "metadatas", "distances"],
)

docs = res.get("documents", [[]])[0]
metas = res.get("metadatas", [[]])[0]
dists = res.get("distances", [[]])[0]

print("\nTop 10 retrieved chunks:")
ranked_rows: List[Dict[str, Any]] = []
for i, (doc, meta, dist) in enumerate(zip(docs, metas, dists), start=1):
    vehicle_id = meta.get("vehicle_id", "unknown") if isinstance(meta, dict) else "unknown"
    brand, model = extract_brand_model(doc or "")
    sim = distance_to_similarity(float(dist))
    ranked_rows.append({
        "rank": i,
        "vehicle_id": vehicle_id,
        "brand": brand,
        "model": model,
        "distance": float(dist),
        "similarity": sim,
        "doc": doc or "",
    })
    print(
        f"{i:2d}. vehicle_id={vehicle_id} | brand={brand} | model={model} "
        f"| distance={dist:.4f} | similarity={sim:.4f}"
    )

# 4) Aggregate by vehicle to avoid chunk duplicates and pick best vehicle
best_by_vehicle: Dict[str, Dict[str, Any]] = {}
for row in ranked_rows:
    vid = str(row["vehicle_id"])
    if vid not in best_by_vehicle or row["distance"] < best_by_vehicle[vid]["distance"]:
        best_by_vehicle[vid] = row

sorted_vehicles = sorted(best_by_vehicle.values(), key=lambda x: x["distance"])
if not sorted_vehicles:
    raise RuntimeError("No retrieval results returned from vector database.")

top_candidate = sorted_vehicles[0]
print("\nTop vehicle candidate:")
print(
    f"vehicle_id={top_candidate['vehicle_id']} | brand={top_candidate['brand']} "
    f"| model={top_candidate['model']} | similarity={top_candidate['similarity']:.4f}"
)

# 5) Ask user to confirm brand and model
confirm = safe_input(
    f"Is this correct? Brand={top_candidate['brand']}, model={top_candidate['model']} (yes/no): ",
    "yes",
).lower()

if confirm not in {"yes", "y"}:
    print("User marked top candidate as not correct. Review top-10 list above for alternatives.")
else:
    print("User confirmed brand/model match.")

# 6) Retrieve complete chunk set for the selected vehicle
selected_vehicle_id = str(top_candidate["vehicle_id"])
all_for_vehicle = collection.get(
    where={"vehicle_id": selected_vehicle_id},
    include=["documents", "metadatas"],
)

vehicle_docs = all_for_vehicle.get("documents", [])
vehicle_metas = all_for_vehicle.get("metadatas", [])

chunk_pairs = []
for d, m in zip(vehicle_docs, vehicle_metas):
    chunk_idx = m.get("chunk_index", 0) if isinstance(m, dict) else 0
    chunk_pairs.append((int(chunk_idx), d))

chunk_pairs.sort(key=lambda x: x[0])
full_text = "\n\n".join([c for _, c in chunk_pairs])

print(f"\nRetrieved {len(chunk_pairs)} chunks for vehicle_id={selected_vehicle_id}.")
print("--- Complete reconstructed text (first 2000 chars) ---")
print(full_text[:2000])

# 7) Optional simple LLM answer to user question: "which brand is it?"
brand_question = "Based on the retrieved record, which brand is the car?"
answer_prompt = f"""
Question: {brand_question}

Retrieved top candidate:
vehicle_id: {top_candidate['vehicle_id']}
brand: {top_candidate['brand']}
model: {top_candidate['model']}
similarity: {top_candidate['similarity']:.4f}

Answer in one sentence.
""".strip()

brand_answer = ollama_chat(answer_prompt, model=CHAT_MODEL)
print("\nLLM answer:")
print(brand_answer)


Search query used:
tesla model y bought 2022 brand tesla

Top 10 retrieved chunks:
 1. vehicle_id=2022 | brand=Škoda | model=Enyaq RS <span style='font-size: 13px;'>(MY24) | distance=305.2007 | similarity=0.0033
 2. vehicle_id=2042 | brand=Nissan | model=Townstar EV Passenger | distance=309.7714 | similarity=0.0032
 3. vehicle_id=2222 | brand=NIO | model=EL8 Standard Range | distance=312.3749 | similarity=0.0032
 4. vehicle_id=2102 | brand=Porsche | model=Taycan Turbo | distance=316.1068 | similarity=0.0032
 5. vehicle_id=2122 | brand=Volvo | model=EX40 Twin Motor Performance <span style='font-size: 13px;'>(MY25) | distance=316.5874 | similarity=0.0031
 6. vehicle_id=2112 | brand=Porsche | model=Taycan Turbo Cross Turismo | distance=318.7585 | similarity=0.0031
 7. vehicle_id=2212 | brand=Kia | model=EV3 Long Range | distance=319.9756 | similarity=0.0031
 8. vehicle_id=2142 | brand=Lotus | model=Emeya S | distance=320.9862 | similarity=0.0031
 9. vehicle_id=1777 | brand=Polestar | mod